# AgoraSim Demo Replay

This notebook replays one archived P4 event window from local `runs/` artifacts. It does not run LLM inference locally. If `runs/` is absent, sync the relevant Vertex/GCS artifacts first.

In [ ]:
from pathlib import Path
import json
import pandas as pd

run = Path('runs/p4/main/oos-2025-g1-tlry-alias-v1')
sim_path = run / 'sim.jsonl'
out_path = run / 'outputs.jsonl'
assert sim_path.exists(), f'Missing {sim_path}; sync archived artifacts first.'

sim = pd.read_json(sim_path, lines=True)
sim[['day', 'ticker', 'flow_imbalance', 'flow_imbalance_unweighted', 'decision_entropy', 'next_day_return']].head(10)

In [ ]:
summary = sim[['flow_imbalance', 'flow_imbalance_unweighted', 'decision_entropy', 'next_day_return']].describe()
summary

In [ ]:
day = sim.sort_values('decision_entropy', ascending=False).iloc[0]['day']
day

In [ ]:
examples = []
with out_path.open(encoding='utf-8') as handle:
    for line in handle:
        row = json.loads(line)
        if str(day)[:10] in str(row.get('request_id', '')) or len(examples) < 5:
            examples.append({
                'request_id': row.get('request_id'),
                'model_id': row.get('model_id'),
                'raw_text': row.get('raw_text'),
            })
        if len(examples) >= 5:
            break
pd.DataFrame(examples)

For final aggregate results, see `docs/RESULTS_REPORT.md`, `docs/RQ3_REPORT.md`, and `docs/P4_FOLLOWUP_REPORT.md`.